In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn nltk gensim torch transformers tqdm --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.6 MB/s eta 0:00:00


In [9]:
!pip install nltk gensim transformers torch scikit-learn wordcloud matplotlib seaborn
!python -m nltk.downloader punkt punkt_tab stopwords


<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
file_path = "/content/data.raw.json"

with open(file_path, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()


,id,title,abstract,keywords,authors,venue,date,teams
0,5096000,Diffusion-based spectral super-resolution of t...,Third octave spectral recording of acoustic se...,"[speech privacy, generative audio, acoustic se...","[Modan Tailleur, Chaymae Benaatia, Mathieu Lag...",[info.info-ai],2025-09-08T00:00:00Z,"[AAU, LS2N]"
1,5113219,Ambiances. A Sensitivity to Ordinary Situations,"How do ambiances shape our sensory, social, an...","[ambiance, atmosphere, ambiances, atmospheres,...","[Jean-Paul Thibaud, Nicolas Tixier, David Zerbib]","[shs.archi, sde.es, shs, shs.art]",2025-09-08T00:00:00Z,[AAU]
2,5129109,Sensory Urban Mobilities: Experiences and Uses...,Beyond documenting all the sensory effects of ...,"[mobilité urbaine, perception sensible, transp...",[Damien Masson],[shs.archi],2025-07-01T00:00:00Z,[AAU]
3,5129157,Preprint_Extended / disabled bodies &amp; atmo...,As much as a symbolic construction or an objec...,"[ambiances, criticism, disabled bodies, forms ...",[Rachel Thomas],[shs],2025-05-11T00:00:00Z,[AAU]
4,5116799,Comparison of GNSS LOS/NLOS Labeling Technique...,The distinction between Line-of-Sight (LOS) an...,"[gnss, satellite visibility, losnlos, sky view...","[Benjamin Beaucamp, Thomas Leduc, Myriam Servi...",[info.info-ts],2025-04-28T00:00:00Z,[AAU]


In [12]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalnum() and t not in stop_words]
    return " ".join(tokens)

df["clean_title"] = df["title"].apply(clean_text)
df["clean_abstract"] = df["abstract"].apply(clean_text)
df.head()


,id,title,abstract,keywords,authors,venue,date,teams,clean_title,clean_abstract,text
0,5096000,Diffusion-based spectral super-resolution of t...,Third octave spectral recording of acoustic se...,"[speech privacy, generative audio, acoustic se...","[Modan Tailleur, Chaymae Benaatia, Mathieu Lag...",[info.info-ai],2025-09-08T00:00:00Z,"[AAU, LS2N]",spectral third octave acoustic sensor data pri...,third octave spectral recording acoustic senso...,diffusionbased spectral superresolution third ...
1,5113219,Ambiances. A Sensitivity to Ordinary Situations,"How do ambiances shape our sensory, social, an...","[ambiance, atmosphere, ambiances, atmospheres,...","[Jean-Paul Thibaud, Nicolas Tixier, David Zerbib]","[shs.archi, sde.es, shs, shs.art]",2025-09-08T00:00:00Z,[AAU],ambiances sensitivity ordinary situations,ambiances shape sensory social built environme...,ambiances sensitivity ordinary situations ambi...
2,5129109,Sensory Urban Mobilities: Experiences and Uses...,Beyond documenting all the sensory effects of ...,"[mobilité urbaine, perception sensible, transp...",[Damien Masson],[shs.archi],2025-07-01T00:00:00Z,[AAU],sensory urban mobilities experiences uses ambi...,beyond documenting sensory effects transportat...,sensory urban mobilities experiences uses ambi...
3,5129157,Preprint_Extended / disabled bodies &amp; atmo...,As much as a symbolic construction or an objec...,"[ambiances, criticism, disabled bodies, forms ...",[Rachel Thomas],[shs],2025-05-11T00:00:00Z,[AAU],disabled bodies amp atmospheres,much symbolic construction object social repre...,preprintextended disabled bodies amp atmospher...
4,5116799,Comparison of GNSS LOS/NLOS Labeling Technique...,The distinction between Line-of-Sight (LOS) an...,"[gnss, satellite visibility, losnlos, sky view...","[Benjamin Beaucamp, Thomas Leduc, Myriam Servi...",[info.info-ts],2025-04-28T00:00:00Z,[AAU],comparison gnss labeling techniques based 3d m...,distinction los nlos signals crucial improving...,comparison gnss losnlos labeling techniques ba...


In [13]:
import nltk
import re
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df["clean_title"] = df["title"].apply(clean_text)
df["clean_abstract"] = df["abstract"].apply(clean_text)
df["text"] = df["clean_title"] + " " + df["clean_abstract"]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
df["venue"] = df["venue"].apply(lambda x: x[0] if isinstance(x, list) and len(x)>0 else "unknown")
from sklearn.preprocessing import LabelEncoder

lbl = LabelEncoder()
df["label"] = lbl.fit_transform(df["venue"])
df["label"].value_counts().head()


,count
label,
270,2427
0,883
97,862
289,839
304,817


In [17]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.